In [ ]:
# conda create -p ./.conda python=3.13.5 ipykernel -y; conda activate ./.conda; python -m ipykernel install --user --name "$((Split-Path -Leaf (Get-Location)))-conda" --display-name "Python ($((Split-Path -Leaf (Get-Location)))-conda)"

# Python conversion from Matlab Shock DOE tool

%reset -f 

import numpy as np
import pandas as pd
import re
import tkinter as tk
from tkinter import filedialog

########################################################################################

# Functions
def parse_damper_id(id_str):
    # Find positions of delimiters
    if not isinstance(id_str, str) or not id_str:
        raise ValueError("id_str must be a non-empty string")
    
    idxUnder = [m.start() for m in re.finditer(r'_', id_str)]
    idxDash = [m.start() for m in re.finditer(r'-', id_str)]
    idxDot = [m.start() for m in re.finditer(r'\.', id_str)]
    if len(idxDash) < 3 or len(idxDot) < 2:
        raise ValueError("id_str format is incorrect")

    lsc = int(id_str[idxDot[0]+1:idxDash[0]])
    hsc = int(id_str[idxDash[0]+1:idxDash[1]])
    lsr = int(id_str[idxDot[-1]+1:idxDash[-1]])
    hsr = int(id_str[idxDash[-1]+1:])

    compValve = id_str[0:idxDot[0]-2]
    blowoff = id_str[idxDash[1]+1:idxUnder[0]]
    rebValve = id_str[idxUnder[0]+1:idxDot[-1]-2]

    compSpring = id_str[idxDot[0]-2:idxDot[0]]
    rebSpring = id_str[idxDot[-1]-2:idxDot[-1]]

    return compValve, compSpring, lsc, hsc, blowoff, rebValve, rebSpring, lsr, hsr

########################################################################################

# Main script
# clear variables

# ask user for damper ID from clipboard
# Read clipboard data and convert to DataFrame

ex = """H4750.22-40-0_H4750.13-40	G4750.25-39-0_G4750.09-36	G4750.45-25-0_G4750.28-31
H4750.18-33-0_H4720.14-13	G4750.13-37-0_G4750.10-39	G4750.29-30-0_G4750.20-34
H4750.21-24-0_H4720.26-36	G4750.22-38-0_G4750.60-40	G4750.20-36-0_G4750.45-40
H4750.16-24-0_H4750.12-32	G4750.32-28-0_G4750.22-37	G4750.23-33-0_G4750.17-36"""
clipboard_data = ex

# clipboard_data = input("Please copy the desired data, then press Enter to continue...")

if clipboard_data is None:
    raise ValueError("No data found in clipboard")
df_clipboard = pd.DataFrame([x.split() for x in clipboard_data.splitlines()]) # split by whitespace

num_ids = np.size(df_clipboard, 1)
num_baseline = num_ids//4
if num_baseline == 0:
    num_baseline = np.size(df_clipboard, 1)
num_rows = np.size(df_clipboard, 0)

if (num_ids%4) != 0:
    if num_rows != 4:
        raise ValueError("Expected 4 columns for damper IDs")

arr = df_clipboard.to_numpy().ravel()          # view when possible, faster and no unnecessary copy
if arr.size%4 != 0 or arr.size not in (4,8,12,16):
    raise ValueError(f"expected 4/8/12/16 elements to reshape to (4,3), got {arr.size}")
arr_reshaped = arr.reshape(4,num_baseline)  # reshape to 4 rows, num_baseline columns
damperInput = arr_reshaped

holdSplit = [None] * 4
splitInput = []
for c in range(num_baseline):
    for n in range(4):
        holdSplit[n] = parse_damper_id(str(damperInput[n, c]))
    splitInput.append(np.array(holdSplit).ravel())
splitInput = np.array(splitInput)

############## Inputs ##############
iterateProps = False
iterateClicks = True
iterateBlowoff = False
forceSymmetry = True

numRand = 2500
clickDelta = 15

valveOpt = ['H47', 'G47', 'G67']
springOpt = [20, 30, 40, 50]
lsRange = [1, 60]
hsRange = [1, 40]
blowRange = [1, 50]

typeKey = ['CompValve', 'CompSpring', 'LSC', 'HSC', 'Blowoff', 'RebValve', 'RebSpring', 'LSR', 'HSR']

randColl = np.empty((numRand, np.size(splitInput,1)), dtype=object) # preallocate
k = 0
for n in range(0, np.size(splitInput,1)): # iterate over each column of splitInput
    
    # if k > np.size(splitInput,1) then k = k - np.size(splitInput,1)
    if k >= np.size(typeKey):
        k -= np.size(typeKey)

    currVar = splitInput[0,n]   # first row, nth column
    currType = typeKey[k] # integer division to get typeKey index
    if currType in ['CompValve', 'RebValve']: # valve types
        if currVar[0:3] not in valveOpt: # check if valve type is valid
            raise ValueError(f"Unexpected valve type: {currVar[0:3]}")
        else: # valid valve type, generate random valves
            currValve = currVar[0:3]
            if iterateProps:
                randValves = np.random.choice(valveOpt, numRand)
            else:
                randValves = np.array([currValve] * numRand)
        randColl[:,n] = randValves
    elif currType in ['CompSpring', 'RebSpring']:
        if int(currVar[-2:]) not in springOpt:
            raise ValueError(f"Unexpected spring size: {currVar[-2:]}")
        else:
            currSpring = int(currVar[-2:])
            if iterateProps:
                randSprings = np.random.choice(springOpt, numRand)
            else:
                randSprings = np.array([currSpring] * numRand)
        randColl[:,n] = randSprings
    elif currType in ['LSC', 'LSR']:
        currClick = int(currVar)
        if (clickDelta + currClick) > lsRange[1]:
            lsUpper = lsRange[1]
        else:
            lsUpper = currClick + clickDelta
        if (currClick - clickDelta) < lsRange[0]:
            lsLower = lsRange[0]
        else:
            lsLower = currClick - clickDelta
        if iterateClicks:
            randLS = np.random.randint(lsLower, lsUpper+1, numRand) # upper bound is exclusive
        else:
            randLS = np.array([currClick] * numRand)
        randColl[:,n] = randLS
    elif currType in ['HSC', 'HSR']:
        currClick = int(currVar)
        if (clickDelta + currClick) > hsRange[1]:
            hsUpper = hsRange[1]
        else:
            hsUpper = currClick + clickDelta
        if (currClick - clickDelta) < hsRange[0]:
            hsLower = hsRange[0]
        else:
            hsLower = currClick - clickDelta
        if iterateClicks:
            randHS = np.random.randint(hsLower, hsUpper+1, numRand) # upper bound is exclusive
        else:
            randHS = np.array([currClick] * numRand)
        randColl[:,n] = randHS
    elif currType == 'Blowoff':
        
        currClick = int(currVar)
        if (clickDelta + currClick) > blowRange[1]: # still working on blowoff, shouldn't be iterated yet
            blowUpper = blowRange[1]
        else:
            blowUpper = currClick + clickDelta
        if (currClick - clickDelta) < blowRange[0]:
            blowLower = blowRange[0]
        else:
            blowLower = currClick - clickDelta
        if iterateBlowoff:
            randBlow = np.random.randint(blowLower, blowUpper+1, numRand) # upper bound is exclusive
        else:
            randBlow = np.array([currClick] * numRand)
        randColl[:,n] = randBlow
    k += 1
       
baseStr = ["Damper Compression High Spd Valve", "Damper Compression High Spd Spring", "Damper Compression Low Spd Clicks",
           "Damper Compression High Spd Clicks", "Damper Compression Blowoff Click", "Damper Rebound High Spd Valve",
           "Damper Rebound High Spd Spring", "Damper Rebound Low Spd Clicks", "Damper Rebound High Spd Clicks"]

lfStr = [s + " LF" for s in baseStr]
rfStr = [s + " RF" for s in baseStr]
lrStr = [s + " LR" for s in baseStr]
rrStr = [s + " RR" for s in baseStr]

colNames = lfStr + rfStr + lrStr + rrStr

idxLF = [n for n in range(len(colNames)) if "LF" in colNames[n]]
idxRF = [n for n in range(len(colNames)) if "RF" in colNames[n]]
idxLR = [n for n in range(len(colNames)) if "LR" in colNames[n]]
idxRR = [n for n in range(len(colNames)) if "RR" in colNames[n]]

if forceSymmetry:
    randColl[:,idxRF] = randColl[:,idxLF]
    randColl[:,idxRR] = randColl[:,idxLR]


batchOutput = np.vstack([splitInput, randColl]) # concatenate baseline and random designs

rr_LSR = batchOutput[:,-2].astype(int)
rr_HSR = batchOutput[:,-1].astype(int)
minRR_LSR = np.min(rr_LSR)
maxRR_HSR = np.max(rr_HSR)

print(f"Min RR LSR: {minRR_LSR}, Max RR HSR: {maxRR_HSR}")




df_output = pd.DataFrame(batchOutput, columns=colNames)
# df_output.to_clipboard(index=False) # copy to clipboard without index
# use file dialog to save
root = tk.Tk()
root.withdraw()  # Hide the root window
file_path = filedialog.asksaveasfilename(defaultextension=".csv",
                                           filetypes=[("CSV files", "*.csv"),
                                                      ("All files", "*.*")])
if file_path:
    df_output.to_csv(file_path, index=False) # save to specified path without index
    print(f"Data saved to {file_path}")
